In [ ]:
from pathlib import Path
import cv2
import pandas as pd
import pytesseract
import os
import shutil


_tesseract = os.environ.get("TESSERACT_CMD") or shutil.which("tesseract")
if not _tesseract:
    for _candidate in (r"C:\Program Files\Tesseract-OCR\tesseract.exe", r"C:\Program Files (x86)\Tesseract-OCR\tesseract.exe"):
        if Path(_candidate).is_file():
            _tesseract = _candidate
            break
if _tesseract:
    pytesseract.pytesseract.tesseract_cmd = _tesseract

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
image_files = sorted((PROJECT_ROOT / "data").rglob("*.png")) + sorted((PROJECT_ROOT / "data").rglob("*.jpg"))

def calculate_ocr_confidence(image_path):
    image = cv2.imread(str(image_path))
    if image is None:
        raise ValueError(f"Image not found or unreadable: {image_path}")
    data = pytesseract.image_to_data(image, output_type=pytesseract.Output.DATAFRAME)
    valid = data[(data["conf"] >= 0) & data["text"].notna() & (data["text"].str.strip() != "")]
    return 0.0 if valid.empty else round(valid["conf"].mean(), 2)

if not image_files:
    print("No image samples yet. Add a PNG or JPG under data/ to calculate OCR confidence.")
else:
    confidence = calculate_ocr_confidence(image_files[0])
    print("OCR confidence:", confidence, "%")
